# TweetAge · 02 Experimentos

Este notebook reúne, en orden, los experimentos que llevaron a la mezcla actual y deja cada resultado en el registro de experimentos (`experiments/registry.csv`).

**Regla de la competencia:** solo modelos de lenguaje propios entrenados con los datos del reto (n-gramas de caracteres y palabras). Nada de vectorización (BoW, TF-IDF), embeddings, transformers ni modelos preentrenados.

**Estructura**

1. Datos y estrategia de validación.
2. Canales: cada modelo de lenguaje que aporta probabilidades.
3. Construcción incremental de la mezcla, canal por canal.
4. Experimento: orden del LM secuencial de palabras (2 vs 3).
5. Resultado por clase de la configuración final.
6. Registro de experimentos.

Los modelos y la mezcla viven en el paquete `tweetage` (`src/tweetage/`); aquí solo se usan. Este notebook no genera submission.

In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

from tweetage import cache, paths, registry, validation
from tweetage import models as M
from tweetage import pipeline as P

CLASES = P.CLASES
SEED_AJUSTE = 20261108        # se usa para elegir hiperparámetros
SEED_CONFIRMACION = 20261109  # partición independiente para confirmar la decisión
SEEDS = (SEED_AJUSTE, SEED_CONFIRMACION)

## 1. Datos y estrategia de validación

Hay textos repetidos en el train (el mismo tweet, a veces con otra etiqueta). Si un texto quedara a la vez en entrenamiento y validación, el modelo lo "recordaría" y la métrica saldría inflada. Por eso la partición se hace con **`StratifiedGroupKFold`**:

- **Grupo** = el texto normalizado (sin espacios en los extremos y en minúsculas): todas las copias de un texto caen del mismo lado.
- **Estratificado** por rango de edad, para mantener la proporción de clases.
- Se toma uno de 5 folds como validación (≈20 %).

La partición está en `tweetage.validation`, para que el análisis de errores (notebook 03) use exactamente la misma.

Se usan **dos semillas**. Con la de *ajuste* se eligen hiperparámetros; con la de *confirmación* se comprueba la decisión en datos que no se usaron para elegir. Un cambio solo se acepta si mejora en ambas.

In [ ]:
train, test = paths.load_data()
X = paths.texts(train)
y = train["age_range"].to_numpy()
print(f"Filas train: {len(X):,} | textos únicos: {len(set(validation.groups_of(X))):,} | filas test: {len(test):,}")
display(train["age_range"].value_counts().reindex(CLASES).to_frame("tweets"))

In [ ]:
# validation.split verifica que ningún texto quede en ambos lados y cachea los índices
splits = {seed: validation.split(X, y, seed) for seed in SEEDS}
for seed, (Xtr, _, Xva, _) in splits.items():
    print(f"Seed {seed}: train={len(Xtr):,} valid={len(Xva):,} textos compartidos=0")

## 2. Canales

Cada canal es un modelo de lenguaje **condicional a la clase**: estima qué tan probable es el texto si lo escribió alguien de cada rango de edad, y lo convierte en probabilidades con una softmax (la temperatura controla qué tan "seguras" son).

| Canal | Modelo | Qué captura | α | Temp. |
|---|---|---|---|---|
| `word1` | Naive Bayes multinomial, unigramas de palabras | vocabulario | 0.01 | 1 |
| `char4`, `char5`, `char6` | Naive Bayes multinomial, n-gramas de caracteres | ortografía, abreviaturas, emojis, alargamientos | 0.1 / 0.1 / 0.03 | 1 |
| `hashtags` | categórico sobre hashtags únicos | temas | 0.1 | 1 |
| `mentions` | categórico sobre menciones únicas | a quién le hablan | 0.1 | 0.35 (en la mezcla) |
| `position` | multinomial sobre (zona, token): primer y último cuarto del tweet | cómo empiezan y terminan los mensajes | 0.03 | 1 |
| `mention_pos` | categórico sobre (mención inicial / dentro del texto, usuario) | respuestas vs. menciones | 0.01 | 1 |
| `char_seq2` | LM de Markov de caracteres, contexto de 2 caracteres | secuencia de caracteres | 0.1 | 0.05 |
| `word_seq2/3` | LM de Markov de palabras, contexto de 1 o 2 palabras | secuencia de palabras | 0.1 | grilla |

Las configuraciones están en `pipeline.CHANNELS`. Las probabilidades de cada canal se guardan en caché: la primera ejecución entrena todo; las siguientes cargan del disco.

Para el experimento de la sección 4 los LMs de palabras se guardan a temperatura 1; luego se re-escalan a cualquier temperatura sin reentrenar, porque softmax(log p / T) = softmax(puntaje / T).

In [ ]:
# Canales extra para la sección 4: LMs de palabras de orden 2 y 3 a temperatura 1.
CHANNELS = {
    **P.CHANNELS,
    "word_seq2_t1": (M.WordSequenceLM, {"alpha": 0.1, "min_count": 3, "order": 2}, 1.0),
    "word_seq3_t1": (M.WordSequenceLM, {"alpha": 0.1, "min_count": 3, "order": 3}, 1.0),
}

probs, masks = {}, {}
for seed, (Xtr, ytr, Xva, _) in splits.items():
    t0 = time.perf_counter()
    probs[seed] = {name: cache.channel(name, Xtr, ytr, Xva, tag=f"seed{seed}", channels=CHANNELS) for name in CHANNELS}
    masks[seed] = M.PositionAwareMentionLM.masks(Xva)["all"]
    print(f"Seed {seed}: {len(CHANNELS)} canales en {time.perf_counter() - t0:.1f} s")

In [ ]:
def evaluar(prob, y_true):
    pred = np.asarray(CLASES)[prob.argmax(axis=1)]
    return f1_score(y_true, pred, average="macro"), accuracy_score(y_true, pred)


def con_temperatura(p_t1, temperature):
    # p_t1 = softmax(puntaje)  =>  softmax(log(p_t1) / T) = softmax(puntaje / T)
    return M.softmax(np.log(np.clip(p_t1, 1e-300, 1.0)) / temperature)


# F1 de cada canal por separado, como referencia
filas = []
for name in P.CHANNELS:
    f1s = [evaluar(probs[s][name], splits[s][3])[0] for s in SEEDS]
    filas.append({"canal": name, "f1_ajuste": f1s[0], "f1_confirmacion": f1s[1]})
pd.DataFrame(filas).set_index("canal").round(4)

## 3. Construcción incremental de la mezcla

La mezcla se armó agregando un canal a la vez, como promedio ponderado de probabilidades. Cada paso se conservó solo si mejoraba en validación. Los pesos son los ya validados (`pipeline.blend_steps`):

1. **`char_mix`**: palabras + caracteres 4/5/6 (pesos 0.216 / 0.144 / 0.240 / 0.400).
2. **`hashtags`**: se agrega el canal de hashtags (0.36 · char_mix + 0.40 · hashtags, renormalizado).
3. **`mentions`**: 70 % lo anterior + 30 % menciones, con temperatura 0.35.
4. **`position`**: 90 % + 10 % canal posicional.
5. **`mention_pos`**: solo en tweets con al menos una mención, 70 % + 30 % menciones con posición.
6. **`char_seq2`**: 85 % + 15 % LM secuencial de caracteres.
7. **`word_seq3`**: 80 % + 20 % LM secuencial de palabras de orden 3 (ver sección 4).

Las celdas siguientes recalculan cada paso en las dos semillas y lo registran.

In [ ]:
DESCRIPCION = {
    "char_mix": "word1 + char4/5/6",
    "hashtags": "+ hashtags",
    "mentions": "+ menciones (T=0.35), peso 0.30",
    "position": "+ posición start_end, peso 0.10",
    "mention_pos": "+ menciones con posición, peso 0.30 solo en tweets con mención",
    "char_seq2": "+ LM secuencial de caracteres orden 2 (T=0.05), peso 0.15",
    "word_seq3": "+ LM secuencial de palabras orden 3 (T=0.2), peso 0.20",
}

pasos = {s: P.blend_steps(probs[s], masks[s]) for s in SEEDS}
filas = []
for i, step in enumerate(pasos[SEED_AJUSTE], start=1):
    nombre = f"mezcla_{i}_{step}"
    fila = {"paso": nombre, "descripcion": DESCRIPCION[step]}
    for s, col in zip(SEEDS, ("ajuste", "confirmacion"), strict=True):
        f1, acc = evaluar(pasos[s][step], splits[s][3])
        fila[f"f1_{col}"] = f1
        registry.log(nombre, {"paso": step, "descripcion": DESCRIPCION[step]}, s, f1, acc,
                     notas="mezcla incremental")
    filas.append(fila)

tabla_mezcla = pd.DataFrame(filas).set_index("paso")
tabla_mezcla["delta_ajuste"] = tabla_mezcla["f1_ajuste"].diff()
tabla_mezcla["delta_confirmacion"] = tabla_mezcla["f1_confirmacion"].diff()
tabla_mezcla.round({c: 4 for c in tabla_mezcla.columns if c.startswith(("f1", "delta"))})

Las mejoras por paso son pequeñas (milésimas de F1), por eso se exige que sean consistentes en las dos semillas antes de gastar un envío.

## 4. Experimento: orden del LM secuencial de palabras

**Pregunta:** ¿conviene un LM de palabras con contexto de una palabra (orden 2) o de dos (orden 3)?

**Base:** la mezcla hasta `char_seq2`, sin ningún LM de palabras.

**Procedimiento**

1. En la semilla de ajuste, para cada orden se prueba una grilla de temperatura × peso: mezcla = (1 − peso) · base + peso · LM.
2. Se elige la mejor configuración **de cada orden** por F1 macro.
3. Ambas se evalúan en la semilla de confirmación y se comparan contra la base y entre sí.

In [ ]:
TEMPERATURAS = (0.05, 0.1, 0.2, 0.3)
PESOS = (0.05, 0.10, 0.15, 0.20)


def mezcla_palabras(seed, orden, temperatura, peso):
    base = pasos[seed]["char_seq2"]
    lm = con_temperatura(probs[seed][f"word_seq{orden}_t1"], temperatura)
    return (1 - peso) * base + peso * lm


yva = splits[SEED_AJUSTE][3]
filas = [{"orden": None, "temperatura": None, "peso": 0.0,
          "f1_macro": evaluar(pasos[SEED_AJUSTE]["char_seq2"], yva)[0]}]
for orden in (2, 3):
    for t in TEMPERATURAS:
        for w in PESOS:
            f1, _ = evaluar(mezcla_palabras(SEED_AJUSTE, orden, t, w), yva)
            filas.append({"orden": orden, "temperatura": t, "peso": w, "f1_macro": f1})

grilla = pd.DataFrame(filas).sort_values("f1_macro", ascending=False).reset_index(drop=True)
grilla.head(12).round(4)

In [ ]:
# Mejor configuración de cada orden en la semilla de ajuste
mejores = {}
for orden in (2, 3):
    fila = grilla[grilla["orden"] == orden].iloc[0]
    mejores[orden] = (float(fila["temperatura"]), float(fila["peso"]))
    print(f"Orden {orden}: temperatura={mejores[orden][0]}, peso={mejores[orden][1]}, F1 ajuste={fila['f1_macro']:.4f}")

In [ ]:
# Confirmación en la partición independiente
yva_c = splits[SEED_CONFIRMACION][3]
f1_base_c, acc_base_c = evaluar(pasos[SEED_CONFIRMACION]["char_seq2"], yva_c)
filas = [{"variante": "base (hasta char_seq2)", "f1_ajuste": grilla.query("peso == 0")["f1_macro"].iloc[0],
          "f1_confirmacion": f1_base_c}]

for orden, (t, w) in mejores.items():
    nombre = f"lm_palabras_orden{orden}"
    config = {"orden": orden, "temperatura": t, "peso": w, "alpha": 0.1, "min_count": 3, "base": "mezcla_6_char_seq2"}
    fila = {"variante": f"{nombre} (T={t}, peso={w})"}
    for s, col in zip(SEEDS, ("ajuste", "confirmacion"), strict=True):
        f1, acc = evaluar(mezcla_palabras(s, orden, t, w), splits[s][3])
        fila[f"f1_{col}"] = f1
        registry.log(nombre, config, s, f1, acc, notas="experimento orden LM palabras")
    filas.append(fila)

confirmacion = pd.DataFrame(filas).set_index("variante")
confirmacion["delta_vs_base_confirmacion"] = confirmacion["f1_confirmacion"] - f1_base_c
confirmacion.round(4)

**Decisión:** se adopta el LM de palabras de **orden 3** con la configuración elegida en la semilla de ajuste, porque supera al orden 2 y a la base en ambas particiones. Es la configuración de `pipeline.CHANNELS["word_seq3"]` y `pipeline.WORD_SEQ_WEIGHT`.

La celda siguiente comprueba que la mezcla final del paquete coincide con la del experimento.

In [ ]:
t3, w3 = mejores[3]
assert (t3, w3) == (P.CHANNELS["word_seq3"][2], P.WORD_SEQ_WEIGHT), "La grilla eligió otra configuración que la del paquete"
for s in SEEDS:
    np.testing.assert_allclose(mezcla_palabras(s, 3, t3, w3), P.blend(probs[s], masks[s]), atol=1e-12)
print("La mezcla del paquete coincide con la elegida en el experimento.")

## 5. Resultado por clase de la configuración final

Evaluado en la semilla de confirmación. Las clases centrales (18-24, 25-34) son las más difíciles: sus estilos de escritura se parecen entre sí y a las clases vecinas.

In [ ]:
pred_final = np.asarray(CLASES)[P.blend(probs[SEED_CONFIRMACION], masks[SEED_CONFIRMACION]).argmax(axis=1)]
print(classification_report(yva_c, pred_final, labels=CLASES, digits=3, zero_division=0))

## 6. Registro de experimentos

Cada experimento queda con su configuración, semilla, F1, accuracy y, cuando existe, el score público de Kaggle. Para anotar el score de una submission nueva: `registry.set_kaggle_score(id, score)`.

In [ ]:
registry.show()